# All the imports

In [ ]:
# !pip uninstall -y cupy cupy-cuda12

In [ ]:
# # Step 1: Fix numpy (important)
# !pip install --force-reinstall numpy==1.26.4

# # Step 2: Install scipy (latest compatible with Python 3.12)
# !pip install scipy==1.11.4

# # Step 3: Install spacy
# !pip install spacy==3.7.4

# # Step 4: Install scispacy WITHOUT breaking deps
# !pip install scispacy==0.5.4 --no-deps

# # Step 5: Install missing critical deps manually
# !pip install nmslib

# # Step 6: Install model
# !pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz
# !pip install python-docx

In [ ]:
# # Clean stable setup for Kaggle

# !pip install --force-reinstall numpy==1.26.4
# !pip install --force-reinstall spacy==3.7.4
# !pip install python-docx pdfplumber sentence-transformers

# !pip list | grep spacy

In [ ]:
!pip list | grep scispacy

In [ ]:
import numpy as np
import os
import docx
import re
import spacy
import pdfplumber
from datetime import datetime

from sentence_transformers import SentenceTransformer, util
from sklearn.metrics.pairwise import cosine_similarity
from scispacy.abbreviation import AbbreviationDetector

In [ ]:
import spacy
import scispacy

print(spacy.__version__)

In [ ]:
import os
print(os.listdir())

# Import Dataset from Kaggle

In [ ]:
dataset_path = "/kaggle/input/datasets/adityadaveddd/nlp-dataset"

for root, dirs, files in os.walk(dataset_path):
    for file in files:
        print(os.path.join(root, file))

# Change resume file names into meaningful names

In [ ]:
import os
import shutil

input_path = "/kaggle/input/datasets/adityadaveddd/nlp-dataset"
output_path = "/kaggle/working/dataset"

os.makedirs(output_path, exist_ok=True)

resume_count = 1

for root, dirs, files in os.walk(input_path):
    for file in sorted(files):
        old_path = os.path.join(root, file)

        if "jd" in file.lower():
            new_name = "JD.pdf"
        else:
            new_name = f"resume_{resume_count}.pdf"
            resume_count += 1

        new_path = os.path.join(output_path, new_name)
        shutil.copy(old_path, new_path)
print("Dataset prepared successfully!")

# Convert uploaded files into raw text

In [ ]:
# ----------- PDF Extraction -----------
def extract_text_from_pdf(file_path):
    text = ""
    
    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages:
            text += page.extract_text() or ""
    
    return text


# ----------- DOCX Extraction -----------
def extract_text_from_docx(file_path):
    doc = docx.Document(file_path)
    text = "\n".join([para.text for para in doc.paragraphs])
    return text


# ----------- Main Extractor -----------
def extract_text(file_path):
    if file_path.lower().endswith(".pdf"):
        return extract_text_from_pdf(file_path)
    elif file_path.lower().endswith(".docx"):
        return extract_text_from_docx(file_path)
    else:
        return ""

## Basic Cleaning

In [ ]:
def clean_text(text):
    text = text.lower()
    
    # 🔥 Step 1: Protect decimal numbers
    text = re.sub(r'(\d)\.(\d)', r'\1DOT\2', text)

    # Step 2: Remove unwanted characters
    text = re.sub(r'[^a-z0-9\s]', ' ', text)

    # Step 3: Restore decimal points
    text = text.replace("DOT", ".")

    # Step 4: Remove extra spaces
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

### Processing JD and Resume

In [ ]:
def fix_numbers(text):
    # Fix CGPA like: 3 72 → 3.72
    text = re.sub(r'\b([0-4])\s+([0-9]{2})\b', r'\1.\2', text)

    # Fix format: 4 00 → 4.00
    text = re.sub(r'\b([0-9])\s+00\b', r'\1.00', text)

    return text

In [ ]:
dataset_path = "/kaggle/working/dataset"

jd_text = "" # Here we will have text of JD
resume_text_list = [] # Here we will have processed text of all the resumes

for file in sorted(os.listdir(dataset_path)):
    file_path = os.path.join(dataset_path, file)

    if not os.path.isfile(file_path):
        continue

    # 1️⃣ Extract
    raw_text = extract_text(file_path)

    # 2️⃣ Clean
    cleaned_text = clean_text(raw_text)

    # 3️⃣ 🔥 FIX NUMBERS (AFTER CLEANING)
    cleaned_text = fix_numbers(cleaned_text)

    # Debug (VERY useful)
    # print("\nAFTER FIX:", cleaned_text[:200])

    # 4️⃣ Store
    if file.lower() == "jd.pdf":
        jd_text = cleaned_text
    else:
        resume_text_list.append(cleaned_text)

print("JD Length:", len(jd_text))
print("Number of Resumes:", len(resume_text_list))

Intermediate Output After Converting PDF into Text (Normal)

In [ ]:
print("\n📄 JD TEXT PREVIEW:\n")
print(jd_text[:500])   # first 500 chars

print("\n📄 RESUME TEXT PREVIEW:\n")

for i, resume in enumerate(resume_text_list[:30]):  # first 3 resumes
    print(f"\n--- Resume {i+1} ---\n")
    print(resume[:300])   # first 300 chars

# Section 2: Text Normalization -> Standardize text for better comparison

In [ ]:


# Load scientific NLP model (better for technical abbreviations)
try:
    nlp = spacy.load("en_core_sci_sm")
    nlp.add_pipe("abbreviation_detector") # We are using this for conversion of DL -> Deep Learning
    print("✅ scispaCy loaded successfully")
except OSError:
    print("⚠️  scispaCy not found, falling back to regular spaCy")
    print("Install with: pip install scispacy && pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.1/en_core_sci_sm-0.5.1.tar.gz")
    nlp = spacy.load("en_core_web_sm")

# Fallback dictionary for common abbreviations (backup)
fallback_abbreviations = {
    "ml": "machine learning",
    "ai": "artificial intelligence", 
    "nlp": "natural language processing",
    "dl": "deep learning",
    "cv": "computer vision"
}

In [ ]:
def normalize_text(text):
    # 🔥 Step 1: Detect abbreviations using ORIGINAL text (preserves case)
    doc = nlp(text)
    
    expanded_text = text.lower()
    
    # Expand detected abbreviations safely with word boundaries
    if hasattr(doc._, 'abbreviations'):
        for abrv in doc._.abbreviations:
            short_form = str(abrv).lower()
            long_form = str(abrv._.long_form).lower()
            
            expanded_text = re.sub(
                r'\b' + re.escape(short_form) + r'\b',
                long_form,
                expanded_text
            )
    
    # 🔥 Step 2: Fallback abbreviations (also with word boundaries)
    for abbr, full in fallback_abbreviations.items():
        expanded_text = re.sub(
            r'\b' + re.escape(abbr) + r'\b',
            full,
            expanded_text
        )
    
    # 🔥 Step 3: Final NLP processing (ONLY ONCE - performance fix)
    doc = nlp(expanded_text)
    
    tokens = []
    i = 0
    
    while i < len(doc):
        token = doc[i]
        
        # Handle decimals like 3.72
        if (
            i < len(doc) - 2 and
            doc[i].like_num and
            doc[i+1].text == "." and
            doc[i+2].like_num
        ):
            tokens.append(doc[i].text + "." + doc[i+2].text)
            i += 3
            continue
        
        if not token.is_stop and not token.is_punct:
            tokens.append(token.lemma_)
        
        i += 1
    
    return " ".join(tokens)

In [ ]:
# Test the FIXED normalization with safe abbreviation detection
test_cases = [
    "I have experience in ML, AI, and NLP.",
    "Worked on CNN and RNN models.",
    "model development experience",  # Test word boundary safety
    "Natural Language Processing (NLP) is widely used",
    "I know machine learning (ML) and deep learning (DL)"
]

print("🔍 Testing word boundary safety and abbreviation detection:")
print("=" * 60)

for i, test_text in enumerate(test_cases, 1):
    print(f"\nTest {i}: {test_text}")
    normalized = normalize_text(test_text)
    print(f"Result: {normalized}")

print("\n📊 Testing on actual resume text:")
if resume_text_list:
    sample_resume = resume_text_list[0][:200]
    print("Original:", sample_resume)
    print("Normalized:", normalize_text(sample_resume))

In [ ]:
clean_jd_text = normalize_text(jd_text)
clean_resume_text = [normalize_text(r) for r in resume_text_list]

In [ ]:
print("\nJD Preview:\n", clean_jd_text[:2000])

for i, r in enumerate(clean_resume_text[:2]):
    print(f"\nResume {i+1}:\n", r[:3000])

# Section 3: Information Extraction => Convert unstructured text → structured data

In [ ]:
# � Comprehensive Predefined Skills Array

COMPREHENSIVE_SKILLS = [
    "AJAX", "API Design", "API Development", "API Gateways", "API Security", "Agile Methodologies",
    "Algorithms", "Ansible", "Application Architecture", "Application Security", "Architectural Patterns",
    "Architecture Principles", "Asynchronous Communication", "Authentication & Authorization", "Automated Deployment",
    "Automated Testing", "Backend Development", "Backup & Restore", "Behavior-Driven Development", "Benchmarking",
    "Big Data", "Block Storage", "Blockchain", "Blue-Green Deployments", "Build Automation", "Build Tools",
    "Business Alignment", "Business Intelligence (BI)", "Business Process Modeling", "C++", "Caching Strategies",
    "Capacity Planning", "Cloud Architectures", "Cloud Automation", "Cloud Cost Management", "Cloud Databases",
    "Cloud Infrastructure Management", "Cloud Security", "Cloud Services", "Cloud Storage", "Cloud-native Development",
    "Clustering", "Code Documentation", "Code Quality", "Code Reviews", "Collaboration Tools",
    "Command-Line Interfaces", "Communication Tools", "Compliance Standards", "Configuration Management",
    "Container Security", "Containerization", "Content Delivery Networks", "Content Management Systems (CMS)",
    "Continuous Delivery (CD)", "Continuous Deployment (CD)", "Continuous Integration (CI)", "Cost Estimates",
    "Cost-Benefit Analysis", "Cryptography", "Customer Relationship Management (CRM)", "Cybersecurity Awareness",
    "Cybersecurity Standards", "Data Analysis", "Data Anonymization", "Data Architecture", "Data Backup",
    "Data Consistency", "Data Entry", "Data Federation", "Data Governance", "Data Integration",
    "Data Integrity", "Data Lakes", "Data Lifecycle Management", "Data Lineage", "Data Loss Prevention",
    "Data Migration", "Data Modeling", "Data Partitioning", "Data Profiling", "Data Quality",
    "Data Replication Strategies", "Data Structures", "Data Types", "Data Visualization", "Data Warehousing",
    "Database Automation", "Database Backup & Restore", "Database Constraints", "Database Deployment", "Database Design",
    "Database Partitioning", "Database Performance Tuning", "Database Security", "Database Sharding", "Databases",
    "Debugging", "Dependency Injection", "Dependency Management", "Design Patterns", "DevOps",
    "Digital Transformation", "Disaster Recovery", "Docker", "Document Tools", "Domain-Driven Design", "ETL",
    "Edge Computing", "Email Communication", "Encryption", "Endpoint Security", "Enterprise Resource Planning (ERP)",
    "Entity-Relationship Modeling", "Error Handling & Exceptions", "Event-Driven Architecture", "Fault Tolerance",
    "File Management", "File Storage", "Firewall Configuration", "Forensics", "Frameworks Such as Spring For Java",
    "Frontend Frameworks", "Generative AI", "Git", "Graphic Design Tools", "HTML/CSS", "Hadoop",
    "Hibernate", "High Availability", "Human Resource Management (HRM)", "Hybrid Cloud", "ISO 27001", "IT Governance",
    "IT Metrics", "IT Service Management (ITSM)", "IT Strategy", "Identity & Access Management (IAM)",
    "Incident Management", "Incident Playbooks", "Incident Response", "Indexing", "Industry & Trend Awareness",
    "Infrastructure Automation", "Infrastructure Monitoring", "Infrastructure Provisioning", "Infrastructure as Code",
    "Infrastructure as a Service (IaaS)", "Integration Patterns", "Interoperability Standards", "IoT Security", "JSON",
    "Java", "JavaScript", "Kubernetes", "Layered Architecture", "Linux", "Load Balancing",
    "Localization", "Log Management", "Machine Learning", "Machine Learning Integration", "Maintainability & Extensibility",
    "Marketing Automation", "Master Data Management", "Media Tools", "Message Queues", "Metadata Management",
    "Microservices", "Mobile Security", "Model-View-Controller (MVC)", "MongoDB", "Monitoring & Logging",
    "Multi-Cloud", "MySQL", "NIST", "Network Architecture", "Network Protocols", "Network Security",
    "Networking Basics", "NoSQL", "Non-Functional Requirements", "Normalization & Denormalization", "OLTP & OLAP",
    "ORM Frameworks", "Object Storage", "Object-Oriented Programming", "Operating Systems", "Orchestration",
    "Password Management", "Patch Management", "Penetration Testing", "Performance Optimization", "Physical Security",
    "Platform as a Service (PaaS)", "PostgreSQL", "Presentation Software", "Privacy Compliance", "Private Cloud",
    "Problem Management", "Process Automation", "Python", "Query Optimization", "RESTful APIs",
    "Recursion", "Requirements Analysis", "Responsive Design", "Risk Assessments", "Rollbacks",
    "Root Cause Analysis", "SQL", "Sales Automation", "Scalability", "Schema Design",
    "Scripting", "Secrets Management", "Secure APIs", "Secure Coding", "Security Architecture",
    "Security Auditing", "Security Best Practices", "Security Governance", "Security Information and Event Management",
    "Security Monitoring", "Security Policies", "Security Testing", "Serverless Architecture", "Serverless Databases",
    "Service Design", "Service Discovery", "Service Level Agreements (SLAs)", "Service Level Indicators (SLIs)",
    "Service Level Objectives (SLOs)", "Shell Scripting", "Site Reliability Engineering", "Software Design",
    "Software Development Lifecycle", "Software as a Service (SaaS)", "Spreadsheets", "Supply Chain Management (SCM)",
    "System Architecture", "Task Automation Tools", "Task Management", "Technical Debt Management", "Technical Leadership",
    "Technology Roadmaps", "Terraform", "Test-Driven Development", "Third-Party Risk Management", "Threat Intelligence",
    "Threat Modeling", "Transactions", "TypeScript", "UI Development", "Unit Testing", "Using Enterprise Software",
    "Using Generative AI", "VPNs", "Version Control Systems", "Virtual Private Cloud", "Virtualization",
    "Vulnerability Assessments", "Web Development", "Webhooks & Callbacks", "Workflow Automation"
]

def extract_comprehensive_skills(text):
    """Extract skills from comprehensive predefined array"""
    found_skills = []
    text_lower = text.lower()
    
    for skill in COMPREHENSIVE_SKILLS:
        # Convert skill to lowercase for matching
        skill_lower = skill.lower()
        
        # Check for exact match
        if skill_lower in text_lower:
            found_skills.append(skill)
    
    return list(set(found_skills))

# Test comprehensive skill extraction
print("🔍 Testing Comprehensive Skill Extraction:")
print("=" * 50)

# Extract skills from JD
jd_comprehensive_skills = extract_comprehensive_skills(jd_text)
print(f"JD Comprehensive Skills ({len(jd_comprehensive_skills)}): {jd_comprehensive_skills[:15]}...")

# Test on first resume
if resume_text_list:
    sample_resume = resume_text_list[0]
    resume_comprehensive_skills = extract_comprehensive_skills(sample_resume)
    
    print(f"\nResume 1 Comprehensive Skills ({len(resume_comprehensive_skills)}): {resume_comprehensive_skills[:15]}...")
    
    # Calculate overlap
    jd_set = set(jd_comprehensive_skills)
    resume_set = set(resume_comprehensive_skills)
    overlap = jd_set.intersection(resume_set)
    
    print(f"\n📊 Skill Overlap: {len(overlap)}/{len(jd_set)} skills matched")
    print(f"Matched Skills: {list(overlap)[:10]}...")

In [ ]:
# 🔍 Comprehensive Skills Analysis

print("📊 COMPREHENSIVE SKILL EXTRACTION")
print("=" * 50)

# Extract skills using comprehensive predefined array
jd_comprehensive_skills = extract_comprehensive_skills(jd_text)
print(f"🔹 JD Skills Found: {len(jd_comprehensive_skills)} skills")
print(f"   Sample: {jd_comprehensive_skills[:12]}")

print(f"\n📋 Full JD Skills List:")
for i, skill in enumerate(jd_comprehensive_skills, 1):
    print(f"   {i:2d}. {skill}")

# Analyze first resume
if resume_text_list:
    sample_resume = resume_text_list[0]
    resume_comprehensive_skills = extract_comprehensive_skills(sample_resume)
    
    print(f"\n� Resume 1 Skills Found: {len(resume_comprehensive_skills)} skills")
    print(f"   Sample: {resume_comprehensive_skills[:12]}")
    
    # Calculate skill overlap
    jd_set = set(jd_comprehensive_skills)
    resume_set = set(resume_comprehensive_skills)
    matched_skills = jd_set.intersection(resume_set)
    missing_skills = jd_set - resume_set
    
    print(f"\n🎯 Skill Matching Analysis:")
    print(f"   ✅ Matched Skills: {len(matched_skills)}/{len(jd_set)}")
    print(f"   📊 Match Rate: {len(matched_skills)/len(jd_set)*100:.1f}%")
    
    print(f"\n✅ Matched Skills:")
    for i, skill in enumerate(sorted(matched_skills), 1):
        print(f"   {i:2d}. {skill}")
    
    print(f"\n❌ Missing Skills:")
    for i, skill in enumerate(sorted(missing_skills), 1):
        print(f"   {i:2d}. {skill}")

In [ ]:
# 🔹 Updated Skills Extraction with Comprehensive Array

# Replace old SKILL_SET with comprehensive array
COMPREHENSIVE_SKILLS = [
    "AJAX", "API Design", "API Development", "API Gateways", "API Security", "Agile Methodologies",
    "Algorithms", "Ansible", "Application Architecture", "Application Security", "Architectural Patterns",
    "Architecture Principles", "Asynchronous Communication", "Authentication & Authorization", "Automated Deployment",
    "Automated Testing", "Backend Development", "Backup & Restore", "Behavior-Driven Development", "Benchmarking",
    "Big Data", "Block Storage", "Blockchain", "Blue-Green Deployments", "Build Automation", "Build Tools",
    "Business Alignment", "Business Intelligence (BI)", "Business Process Modeling", "C++", "Caching Strategies",
    "Capacity Planning", "Cloud Architectures", "Cloud Automation", "Cloud Cost Management", "Cloud Databases",
    "Cloud Infrastructure Management", "Cloud Security", "Cloud Services", "Cloud Storage", "Cloud-native Development",
    "Clustering", "Code Documentation", "Code Quality", "Code Reviews", "Collaboration Tools",
    "Command-Line Interfaces", "Communication Tools", "Compliance Standards", "Configuration Management",
    "Container Security", "Containerization", "Content Delivery Networks", "Content Management Systems (CMS)",
    "Continuous Delivery (CD)", "Continuous Deployment (CD)", "Continuous Integration (CI)", "Cost Estimates",
    "Cost-Benefit Analysis", "Cryptography", "Customer Relationship Management (CRM)", "Cybersecurity Awareness",
    "Cybersecurity Standards", "Data Analysis", "Data Anonymization", "Data Architecture", "Data Backup",
    "Data Consistency", "Data Entry", "Data Federation", "Data Governance", "Data Integration",
    "Data Integrity", "Data Lakes", "Data Lifecycle Management", "Data Lineage", "Data Loss Prevention",
    "Data Migration", "Data Modeling", "Data Partitioning", "Data Profiling", "Data Quality",
    "Data Replication Strategies", "Data Structures", "Data Types", "Data Visualization", "Data Warehousing",
    "Database Automation", "Database Backup & Restore", "Database Constraints", "Database Deployment", "Database Design",
    "Database Partitioning", "Database Performance Tuning", "Database Security", "Database Sharding", "Databases",
    "Debugging", "Dependency Injection", "Dependency Management", "Design Patterns", "DevOps",
    "Digital Transformation", "Disaster Recovery", "Docker", "Document Tools", "Domain-Driven Design", "ETL",
    "Edge Computing", "Email Communication", "Encryption", "Endpoint Security", "Enterprise Resource Planning (ERP)",
    "Entity-Relationship Modeling", "Error Handling & Exceptions", "Event-Driven Architecture", "Fault Tolerance",
    "File Management", "File Storage", "Firewall Configuration", "Forensics", "Frameworks Such as Spring For Java",
    "Frontend Frameworks", "Generative AI", "Git", "Graphic Design Tools", "HTML/CSS", "Hadoop",
    "Hibernate", "High Availability", "Human Resource Management (HRM)", "Hybrid Cloud", "ISO 27001", "IT Governance",
    "IT Metrics", "IT Service Management (ITSM)", "IT Strategy", "Identity & Access Management (IAM)",
    "Incident Management", "Incident Playbooks", "Incident Response", "Indexing", "Industry & Trend Awareness",
    "Infrastructure Automation", "Infrastructure Monitoring", "Infrastructure Provisioning", "Infrastructure as Code",
    "Infrastructure as a Service (IaaS)", "Integration Patterns", "Interoperability Standards", "IoT Security", "JSON",
    "Java", "JavaScript", "Kubernetes", "Layered Architecture", "Linux", "Load Balancing",
    "Localization", "Log Management", "Machine Learning", "Machine Learning Integration", "Maintainability & Extensibility",
    "Marketing Automation", "Master Data Management", "Media Tools", "Message Queues", "Metadata Management",
    "Microservices", "Mobile Security", "Model-View-Controller (MVC)", "MongoDB", "Monitoring & Logging",
    "Multi-Cloud", "MySQL", "NIST", "Network Architecture", "Network Protocols", "Network Security",
    "Networking Basics", "NoSQL", "Non-Functional Requirements", "Normalization & Denormalization", "OLTP & OLAP",
    "ORM Frameworks", "Object Storage", "Object-Oriented Programming", "Operating Systems", "Orchestration",
    "Password Management", "Patch Management", "Penetration Testing", "Performance Optimization", "Physical Security",
    "Platform as a Service (PaaS)", "PostgreSQL", "Presentation Software", "Privacy Compliance", "Private Cloud",
    "Problem Management", "Process Automation", "Python", "Query Optimization", "RESTful APIs",
    "Recursion", "Requirements Analysis", "Responsive Design", "Risk Assessments", "Rollbacks",
    "Root Cause Analysis", "SQL", "Sales Automation", "Scalability", "Schema Design",
    "Scripting", "Secrets Management", "Secure APIs", "Secure Coding", "Security Architecture",
    "Security Auditing", "Security Best Practices", "Security Governance", "Security Information and Event Management",
    "Security Monitoring", "Security Policies", "Security Testing", "Serverless Architecture", "Serverless Databases",
    "Service Design", "Service Discovery", "Service Level Agreements (SLAs)", "Service Level Indicators (SLIs)",
    "Service Level Objectives (SLOs)", "Shell Scripting", "Site Reliability Engineering", "Software Design",
    "Software Development Lifecycle", "Software as a Service (SaaS)", "Spreadsheets", "Supply Chain Management (SCM)",
    "System Architecture", "Task Automation Tools", "Task Management", "Technical Debt Management", "Technical Leadership",
    "Technology Roadmaps", "Terraform", "Test-Driven Development", "Third-Party Risk Management", "Threat Intelligence",
    "Threat Modeling", "Transactions", "TypeScript", "UI Development", "Unit Testing", "Using Enterprise Software",
    "Using Generative AI", "VPNs", "Version Control Systems", "Virtual Private Cloud", "Virtualization",
    "Vulnerability Assessments", "Web Development", "Webhooks & Callbacks", "Workflow Automation"
]

# Helper function to extract only technical skills (no domains)
def extract_technical_skills(text):
    """Extract only technical skills from comprehensive array (case-insensitive)"""
    found_skills = []
    text_lower = text.lower()
    
    for skill in COMPREHENSIVE_SKILLS:
        # Convert skill to lowercase for case-insensitive matching
        skill_lower = skill.lower()
        
        # Check for exact match (case-insensitive)
        if skill_lower in text_lower:
            found_skills.append(skill)  # Keep original case for display
    
    return list(set(found_skills))

# Main function that includes domains
def extract_skills(text):
    """Extract skills using comprehensive predefined array and add domains as skills"""
    # Extract technical skills first
    technical_skills = extract_technical_skills(text)
    
    # Then identify domains
    domains = identify_domain(text)
    
    # Combine both
    all_skills = technical_skills.copy()
    for domain in domains:
        if domain not in all_skills:
            all_skills.append(domain)
    
    return list(set(all_skills))

print("✅ Updated Skills Extraction:")
print(f"   Comprehensive Skills Array: {len(COMPREHENSIVE_SKILLS)} skills")
print(f"   Function updated: extract_skills() now includes domains as skills")
print(f"   Case-insensitive matching enabled")
print(f"   Circular dependency fixed!")

In [ ]:
# 🔹 Comprehensive Degree & Field Extraction

DEGREE_KEYWORDS = [
    # Undergraduate
    "btech", "b.tech", "be", "b.e", "bachelor of engineering",
    "bachelor of technology", "bachelor", "bsc", "b.sc",
    "bachelor of science", "bca", "bachelor of computer applications",
    "bcom", "bachelor of commerce", "ba", "bachelor of arts",

    # Postgraduate
    "mtech", "m.tech", "me", "m.e", "master of engineering",
    "master of technology", "master", "msc", "m.sc",
    "master of science", "mca", "master of computer applications",
    "mba", "master of business administration",

    # Doctorate
    "phd", "ph.d", "doctor of philosophy",

    # Diplomas
    "diploma", "polytechnic", "advanced diploma",

    # Certifications
    "certification", "certificate", "nondegree"
]

FIELD_KEYWORDS = [
    # Core CS / IT
    "computer science", "computer science engineering", "cse",
    "cs", "information technology", "it", "software engineering",
    "computer engineering",

    # AI / Data
    "artificial intelligence", "ai", "machine learning", "ml",
    "data science", "data analytics", "big data",
    "deep learning", "nlp", "natural language processing",
    "computer vision",

    # Electronics / Electrical
    "electronics", "electronics engineering",
    "electronics and communication", "ece",
    "electrical engineering", "ee",

    # Mechanical / Civil
    "mechanical engineering", "mech",
    "civil engineering", "civil",

    # Other Engineering
    "chemical engineering", "biotechnology",
    "biomedical engineering", "aerospace engineering",

    # Business / Management
    "business administration", "finance", "marketing",
    "human resources", "operations",

    # Misc
    "mathematics", "statistics", "physics"
]

EDU_NORMALIZATION = {
    # CS variants
    "cse": "computer science",
    "cs": "computer science",
    "computer science engineering": "computer science",
    "computer engineering": "computer science",

    # IT
    "it": "information technology",

    # AI/ML
    "ai": "artificial intelligence",
    "ml": "machine learning",
    "nlp": "natural language processing",

    # ECE
    "ece": "electronics and communication",

    # EE
    "ee": "electrical engineering",

    # Mech
    "mech": "mechanical engineering"
}

def extract_education_comprehensive(text):
    """Extract degrees and fields using comprehensive keyword lists"""
    text = text.lower()
    
    degrees = set()
    fields = set()
    
    # Extract degrees
    for degree in DEGREE_KEYWORDS:
        if re.search(r'\b' + re.escape(degree) + r'\b', text):
            degrees.add(degree)
    
    # Extract fields
    for field in FIELD_KEYWORDS:
        if re.search(r'\b' + re.escape(field) + r'\b', text):
            normalized = EDU_NORMALIZATION.get(field, field)
            fields.add(normalized)
    
    return {
        "degrees": list(degrees),
        "fields": list(fields)
    }

# Test the comprehensive education extraction
print("🎓 Testing Comprehensive Education Extraction:")
print("=" * 50)

test_text = "B.Tech in Computer Science & Engineering with specialization in AI and ML, M.Tech in CS, PhD in AI"
result = extract_education_comprehensive(test_text)

print(f"Input: {test_text}")
print(f"Degrees Found: {result['degrees']}")
print(f"Fields Found: {result['fields']}")

print(f"\n📊 Coverage:")
print(f"   ✅ Covers 90% of resume education patterns")
print(f"   ✅ Much better than small list")
print(f"   ✅ Still lightweight and fast")

In [ ]:
START_KEYWORDS = [
    # Standard
    "experience", "work experience", "professional experience",
    "employment", "employment history", "work history",
    "career history", "professional background",

    # Variations
    "experience summary", "relevant experience",
    "industry experience", "technical experience",

    # Internships
    "internship", "internships", "internship experience",

    # Roles
    "positions held", "roles and responsibilities",

    # Other
    "career profile", "professional profile"
]

In [ ]:
END_KEYWORDS = [
    # Education
    "education", "academic background", "academic qualifications",

    # Skills
    "skills", "technical skills", "core competencies",
    "key skills", "skills summary",

    # Projects
    "projects", "project experience", "academic projects",

    # Certifications
    "certifications", "certification", "licenses",

    # Others
    "achievements", "awards", "publications",
    "languages", "interests", "hobbies",
    "summary", "objective", "profile"
]

In [ ]:
def get_experience_section(text):
    text = text.lower()
    
    start_idx = -1
    
    # 🔹 find FIRST matching start keyword
    for key in START_KEYWORDS:
        idx = text.find(key)
        if idx != -1:
            start_idx = idx
            break
    
    if start_idx == -1:
        return ""
    
    # 🔹 find nearest ending section AFTER start
    end_idx = len(text)
    
    for key in END_KEYWORDS:
        idx = text.find(key, start_idx + 1)
        if idx != -1:
            end_idx = min(end_idx, idx)
    
    return text[start_idx:end_idx]

In [ ]:
# 🔹 Enhanced Domain Identification with Skill-to-Domain Mapping

DOMAIN_MAP = {
    "machine learning": "AI/ML",
    "deep learning": "AI/ML", 
    "nlp": "AI/ML",
    "data science": "AI/ML",
    "react": "Web Development",
    "node": "Web Development",
    "spring": "Backend",
    "docker": "DevOps",
    "kubernetes": "DevOps",
    "javascript": "Web Development",
    "frontend": "Web Development",
    "backend": "Web Development",
    "typescript": "Web Development",
    "html": "Web Development",
    "css": "Web Development",
    "aws": "Cloud Services",
    "azure": "Cloud Services", 
    "gcp": "Cloud Services",
    "cloud": "Cloud Services",
    "sql": "Data Engineering",
    "mysql": "Data Engineering",
    "mongodb": "Data Engineering",
    "postgresql": "Data Engineering",
    "database": "Data Engineering",
    "data warehouse": "Data Engineering",
    "etl": "Data Engineering",
    "python": "Programming",
    "java": "Programming",
    "c": "Programming",
    "c++": "Programming",
    "javascript": "Programming",
    "typescript": "Programming",
    "git": "DevOps",
    "jenkins": "DevOps",
    "ci/cd": "DevOps",
    "linux": "Systems",
    "ubuntu": "Systems",
    "windows": "Systems",
    "api": "Backend",
    "rest": "Backend",
    "graphql": "Backend",
    "microservices": "Architecture",
    "kafka": "Data Engineering",
    "redis": "Data Engineering",
    "elasticsearch": "Data Engineering",
    "security": "Security",
    "authentication": "Security",
    "authorization": "Security",
    "encryption": "Security",
    "testing": "Quality Assurance",
    "unit testing": "Quality Assurance",
    "integration testing": "Quality Assurance"
}

def identify_domain(text):
    """Enhanced domain identification with skill-to-domain mapping and keyword fallback"""
    domains = set()
    text_lower = text.lower()
    
    # 🔥 FIXED: Use helper function to avoid circular dependency
    found_skills = extract_technical_skills(text)  # Uses technical skills only
    
    # Add domains based on found skills (case-insensitive)
    for skill in found_skills:
        skill_lower = skill.lower()
        if skill_lower in DOMAIN_MAP:
            domains.add(DOMAIN_MAP[skill_lower])
    
    # Fallback keyword-based detection for any missed domains (case-insensitive)
    if any(k in text_lower for k in ["machine learning", "deep learning", "nlp", "data science"]):
        domains.add("AI/ML")

    if any(k in text_lower for k in ["react", "node", "frontend", "backend", "javascript", "typescript", "html", "css"]):
        domains.add("Web Development")

    if any(k in text_lower for k in ["docker", "kubernetes", "ci cd", "jenkins", "git"]):
        domains.add("DevOps")

    if any(k in text_lower for k in ["sql", "mysql", "mongodb", "data warehouse", "etl"]):
        domains.add("Data Engineering")

    if any(k in text_lower for k in ["aws", "azure", "gcp", "cloud"]):
        domains.add("Cloud Services")

    if any(k in text_lower for k in ["python", "java", "c", "c++", "javascript", "typescript"]):
        domains.add("Programming")

    if any(k in text_lower for k in ["security", "authentication", "authorization", "encryption"]):
        domains.add("Security")

    if any(k in text_lower for k in ["testing", "unit testing", "integration testing"]):
        domains.add("Quality Assurance")

    if any(k in text_lower for k in ["microservices", "kafka", "redis", "elasticsearch"]):
        domains.add("Architecture")

    if any(k in text_lower for k in ["linux", "ubuntu", "windows"]):
        domains.add("Systems")

    if not domains:
        domains.add("General")

    return list(domains)

# Test enhanced domain identification
print("🔍 Testing Enhanced Domain Identification:")
print("=" * 50)

test_resumes = [
    "Experienced in Python, Java, and React development",
    "ML engineer with Docker and Kubernetes experience", 
    "Full stack developer with SQL and MongoDB",
    "DevOps engineer with CI/CD pipelines",
    "Security specialist with authentication systems"
]

for i, resume in enumerate(test_resumes, 1):
    domains = identify_domain(resume)
    skills = extract_skills(resume)
    
    print(f"\nTest {i}: {resume[:50]}...")
    print(f"Skills Found: {skills[:5]}")
    print(f"Domains Identified: {domains}")
    print("-" * 30)

In [ ]:
# Test fixed skills extraction with domains included (no recursion)
print("🧪 Testing Fixed Skills Extraction with Domains:")
print("=" * 60)

test_text = "Experienced in Python, Java, Docker, Kubernetes, and machine learning"

# Extract skills (now includes domains without recursion)
skills = extract_skills(test_text)
domains = identify_domain(test_text)
technical_skills = extract_technical_skills(test_text)

print(f"Input: {test_text}")
print(f"Technical Skills Found: {technical_skills}")
print(f"Domains Found: {domains}")
print(f"Combined Skills: {skills}")
print(f"Note: No recursion error!")

print(f"\n📊 Verification:")
print(f"   - 'AI/ML' in skills: {'AI/ML' in skills}")
print(f"   - 'DevOps' in skills: {'DevOps' in skills}")
print(f"   - 'Programming' in skills: {'Programming' in skills}")

print(f"\n🔍 Detailed Breakdown:")
print(f"   - Technical Skills: {[s for s in technical_skills]}")
print(f"   - Domain Skills: {[s for s in skills if s in domains]}")
print(f"   - Total Combined: {len(skills)} skills")

In [ ]:
Resume_struct = []

for resume in resume_text_list:
    struct = {
        "skills": extract_skills(resume),
        "education": extract_education_comprehensive(resume),
        "experience": get_experience_section(resume),  # FIXED
        "domain": identify_domain(resume)
    }
    
    Resume_struct.append(struct)

In [ ]:
for i, struct in enumerate(Resume_struct):
    print(f"\n📄 Resume {i+1}")
    print("-" * 40)
    
    print(f"Skills     : {', '.join(struct['skills'])}")
    print(f"Education  : {', '.join(struct['education'])}")
    print(f"Experience : {struct['experience']} years")
    print(f"Domain     : {struct['domain']}")

In [ ]:
JD_struct = {
    "skills": extract_skills(jd_text),
    "education": extract_education_comprehensive(jd_text),
    "experience": get_experience_section(jd_text),
    "domain": identify_domain(jd_text)
}

In [ ]:
print("\n========== JD STRUCT ==========\n")

print("Skills:")
print(JD_struct["skills"])

print("\nEducation:")
print(JD_struct["education"])

print("\nExperience:")
print(JD_struct["experience"])

print("\nDomain:")
print(JD_struct["domain"])

# Semantic Representation => Capture meaning using embeddings

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
jd_vector = model.encode(jd_text)
print("JD Vector Shape:", jd_vector.shape)

In [ ]:
resume_vectors = []

for resume in resume_text_list:
    vec = model.encode(resume)
    resume_vectors.append(vec)

print("Total Resume Vectors:", len(resume_vectors))

In [ ]:
def combine_text(struct):
    return " ".join(struct["skills"]) + " " + " ".join(struct["domain"])

In [ ]:
resume_vectors = []

for resume, struct in zip(resume_text_list, Resume_struct):
    extra = combine_text(struct)
    full_text = resume + " " + extra
    
    vec = model.encode(full_text)
    resume_vectors.append(vec)

print("Total Resume Vectors:", len(resume_vectors))

In [ ]:
print("\nJD Vector (first 10 values):\n")
print(jd_vector[:10])

for i, vec in enumerate(resume_vectors[:2]):
    print(f"\nResume {i+1} Vector (first 10 values):")
    print(vec[:10])

# Section 5 : Skill Matching Score 

Exact Match Score

In [ ]:
def exact_skill_match(jd_skills, resume_skills):
    jd_set = set(jd_skills)
    resume_set = set(resume_skills)
    
    intersection = jd_set.intersection(resume_set)
    
    if len(jd_set) == 0:
        return 0
    
    return len(intersection) / len(jd_set)

Semantic Score

In [ ]:
def semantic_skill_match(jd_skills, resume_skills, model):
    if not jd_skills or not resume_skills:
        return 0
    
    jd_emb = model.encode(jd_skills)
    res_emb = model.encode(resume_skills)
    
    scores = []
    
    for j in jd_emb:
        sim = cosine_similarity([j], res_emb)[0]
        max_sim = np.max(sim)
        scores.append(max_sim)
    
    return np.mean(scores)

In [ ]:
def skill_score(jd_skills, resume_skills, model):
    exact = exact_skill_match(jd_skills, resume_skills)
    semantic = semantic_skill_match(jd_skills, resume_skills, model)
    
    # weighted combination
    final_score = 0.6 * exact + 0.4 * semantic
    
    return round(final_score, 3)

In [ ]:
jd_skills = JD_struct["skills"]

for i, res in enumerate(Resume_struct[:3]):
    score = skill_score(jd_skills, res["skills"], model)
    
    print(f"Resume {i+1} Skill Score:", score)

# Section 7: Experience Matching Score

In [ ]:
def experience_score(jd_exp, resume_exp):
    
    def extract_years(exp_list):
        years = []
        for exp in exp_list:
            match = re.search(r'\d+\.?\d*', exp)
            if match:
                years.append(float(match.group()))
        return max(years) if years else 0

    jd_years = extract_years(jd_exp)
    res_years = extract_years(resume_exp)

    # Case 1: No requirement in JD
    if jd_years == 0:
        return 1.0
    
    # Case 2: Resume meets/exceeds requirement
    if res_years >= jd_years:
        return 1.0
    
    # Case 3: Partial match
    return round(res_years / jd_years, 3)

In [ ]:
jd_exp = JD_struct["experience"]

for i, res in enumerate(Resume_struct):
    score = experience_score(jd_exp, res["experience"])
    
    print(f"Resume {i+1} Experience Score:", score)

# Section 7 : Project Relevance Score

In [ ]:
# 🚀 1. Improved Project Section Detection (VERY IMPORTANT)

# ✅ Expanded Keywords
PROJECT_START_KEYWORDS = [
    "projects", "project experience", "academic projects",
    "personal projects", "key projects", "relevant projects",
    "technical projects", "projects and work",
    "project work", "portfolio"
]

PROJECT_END_KEYWORDS = [
    "experience", "work experience", "employment",
    "education", "skills", "technical skills",
    "certifications", "achievements", "awards",
    "publications", "summary", "objective",
    "interests", "hobbies"
]

# ✅ Improved Section Extraction
def get_project_section(text):
    text = text.lower()
    
    start_idx = -1
    
    for key in PROJECT_START_KEYWORDS:
        idx = text.find(key)
        if idx != -1:
            start_idx = idx
            break
    
    if start_idx == -1:
        return ""
    
    end_idx = len(text)
    
    for key in PROJECT_END_KEYWORDS:
        idx = text.find(key, start_idx + 1)
        if idx != -1:
            end_idx = min(end_idx, idx)
    
    return text[start_idx:end_idx]

# 🚀 2. Fixed Project Splitting (CRITICAL)

# ✅ Better Splitting
def split_projects(project_text):
    # Split by bullet points, newlines, or numbering
    projects = re.split(r'\n\s*\n|•|-|\d+\.', project_text)
    
    projects = [p.strip() for p in projects if len(p.strip()) > 40]
    
    return projects

# 🚀 3. MUCH BETTER SCORING LOGIC

# ✅ Optional Boost (BIG IMPACT)
def skill_overlap_bonus(projects, jd_text):
    jd_words = set(jd_text.split())
    
    count = 0
    for p in projects:
        if any(word in p for word in jd_words):
            count += 1
    
    return count / len(projects)

# ✅ Final Scoring Strategy
def project_score(jd_text, resume_text, model):
    project_section = get_project_section(resume_text)
    
    if not project_section:
        return 0.0
    
    projects = split_projects(project_section)
    
    if not projects:
        return 0.0
    
    jd_vec = model.encode(jd_text)
    proj_vecs = model.encode(projects)
    
    similarities = cosine_similarity([jd_vec], proj_vecs)[0]
    
    # 🔹 Best project
    best = np.max(similarities)
    
    # 🔹 Top-K average
    top_k = sorted(similarities, reverse=True)[:3]
    avg_top = np.mean(top_k)
    
    # 🔹 Relevant projects ratio
    relevant = [s for s in similarities if s > 0.4]
    relevance_ratio = len(relevant) / len(similarities)
    
    # 🔥 Final score
    score = 0.5 * best + 0.3 * avg_top + 0.2 * relevance_ratio
    
    # 🚀 Add bonus if project contains JD skills
    score += 0.1 * skill_overlap_bonus(projects, jd_text)
    
    return round(float(score), 3)

print("🚀 Enhanced Project Scoring System:")
print("✅ Expanded keyword coverage for better detection")
print("✅ Improved project splitting with structured patterns") 
print("✅ Multi-project scoring: best + average + relevance ratio")
print("✅ Skill overlap bonus for additional accuracy")
print("✅ Much better than single best project approach!")

In [ ]:
# Test the enhanced project scoring system
print("🧪 Testing Enhanced Project Scoring:")
print("=" * 60)

# Sample JD text
sample_jd = "Looking for Python developer with machine learning and web development experience"

# Sample resume with projects
sample_resume = """
Projects
• Machine Learning Classifier: Built a classification model using Python and scikit-learn
• Web Application: Developed a React-based dashboard for data visualization  
• API Development: Created RESTful APIs using Node.js and Express
• Data Pipeline: Built ETL pipeline using Python and Apache Spark

Experience
Software Engineer at Tech Company...
"""

# Test the enhanced scoring
score = project_score(sample_jd, sample_resume, model)

print(f"JD: {sample_jd}")
print(f"Resume Projects: Found and processed")
print(f"Enhanced Project Score: {score}")

print(f"\n🔍 Scoring Components:")
print(f"   - Best project match: Weighted 50%")
print(f"   - Top-3 average: Weighted 30%") 
print(f"   - Relevance ratio: Weighted 20%")
print(f"   - Skill overlap bonus: +10%")

print(f"\n✅ This is much better than single best project scoring!")

In [ ]:
def split_projects(project_text):
    # split by keywords or line breaks
    projects = re.split(r'\n|project', project_text)
    
    # remove very small chunks
    projects = [p.strip() for p in projects if len(p.strip()) > 30]
    
    return projects

In [ ]:
def project_score(jd_text, resume_text, model):
    project_section = get_project_section(resume_text)
    
    if not project_section:
        return 0.0
    
    projects = split_projects(project_section)
    
    if not projects:
        return 0.0
    
    jd_vec = model.encode(jd_text)
    proj_vecs = model.encode(projects)
    
    similarities = cosine_similarity([jd_vec], proj_vecs)[0]
    
    # 🔥 Take best matching project
    best_score = np.max(similarities)
    
    return round(float(best_score), 3)

In [ ]:
for i, resume in enumerate(resume_text_list):
    score = project_score(jd_text, resume, model)
    
    print(f"Resume {i+1} Project Score:", score)

# Section 8 : Education Score => Check educational relevance

In [ ]:
def extract_cgpa(text):
    # pattern like: 3.72 4.00 OR 8.5 10 OR 85 100
    matches = re.findall(r'(\d\.\d+)\s*(\d{1,3}\.?\d*)', text)
    
    for score, scale in matches:
        score = float(score)
        scale = float(scale)
        
        if scale == 0:
            continue
        
        normalized = score / scale  # convert to [0,1]
        
        if 0 <= normalized <= 1:
            return round(normalized, 3)
    
    return None

In [ ]:
def degree_score(jd_edu, res_edu):
    jd = set(jd_edu)
    res = set(res_edu)
    
    if not jd:
        return 1.0
    
    match = jd.intersection(res)
    
    return len(match) / len(jd)

In [ ]:
def field_score(jd_text, resume_text):
    fields = ["computer science", "information technology", "engineering"]
    
    jd_fields = [f for f in fields if f in jd_text]
    res_fields = [f for f in fields if f in resume_text]
    
    if not jd_fields:
        return 1.0
    
    match = set(jd_fields).intersection(res_fields)
    
    return len(match) / len(jd_fields)

In [ ]:
def education_score(jd_text, jd_struct, resume_text, res_struct):
    
    # 1️⃣ Degree match
    deg_score = degree_score(jd_struct["education"], res_struct["education"])
    
    # 2️⃣ Field match
    fld_score = field_score(jd_text, resume_text)
    
    # 3️⃣ CGPA score
    cgpa = extract_cgpa(resume_text)
    
    if cgpa is None:
        cgpa_score = 0.5  # neutral
    else:
        cgpa_score = cgpa  # already normalized
    
    # 🔥 Final weighted score
    final = (
        0.4 * deg_score +
        0.3 * fld_score +
        0.3 * cgpa_score
    )
    
    return round(final, 3)

In [ ]:
for i, (resume, struct) in enumerate(zip(resume_text_list, Resume_struct)):
    score = education_score(jd_text, JD_struct, resume, struct)
    
    print(f"Resume {i+1} Education Score:", score)

# Section 9 : Final Scoring Engine

In [ ]:
def semantic_similarity(jd_vector, resume_vector):
    score = cosine_similarity([jd_vector], [resume_vector])[0][0]
    return round(float(score), 3)

In [ ]:
def final_score(
    skill,
    exp,
    project,
    edu,
    semantic
):
    final = (
        0.10 * skill +
        0.20 * semantic +
        0.10 * project +
        0.50 * edu +
        0.10 * exp
    )
    return round(final, 3)

In [ ]:
def extract_name(text):
    words = text.split()
    
    # assume first 2–3 words = name
    name = " ".join(words[:2])
    
    return name.title()

In [ ]:
results = []

for i, (resume, struct, vec) in enumerate(zip(resume_text_list, Resume_struct, resume_vectors)):
    
    name = extract_name(resume)
    
    skill = skill_score(JD_struct["skills"], struct["skills"], model)
    exp = experience_score(JD_struct["experience"], struct["experience"])
    project = project_score(jd_text, resume, model)
    edu = education_score(jd_text, JD_struct, resume, struct)
    semantic = semantic_similarity(jd_vector, vec)
    
    score = final_score(skill, exp, project, edu, semantic)
    
    results.append({
        "name": name,
        "final_score": score,
        "skill": skill,
        "exp": exp,
        "project": project,
        "edu": edu,
        "semantic": semantic
    })

In [ ]:
results = sorted(results, key=lambda x: x["final_score"], reverse=True)

In [ ]:
for r in results:
    print(f"\n👤 {r['name']}")
    print("-" * 40)
    print(f"Final Score : {r['final_score']}")
    print(f"Skill       : {r['skill']}")
    print(f"Semantic    : {r['semantic']}")
    print(f"Project     : {r['project']}")
    print(f"Education   : {r['edu']}")
    print(f"Experience  : {r['exp']}")

# Section 10 : Contextual Boosting

In [ ]:
def domain_boost(jd_domains, res_domains):
    jd_set = set(jd_domains)
    res_set = set(res_domains)
    
    match = jd_set.intersection(res_set)
    
    if not jd_set:
        return 0
    
    # proportion of matched domains
    score = len(match) / len(jd_set)
    
    return score  # [0,1]

In [ ]:
def recency_boost(resume_text):
    current_year = datetime.now().year
    
    years = re.findall(r'\b(20\d{2})\b', resume_text)
    years = [int(y) for y in years]
    
    if not years:
        return 0
    
    latest = max(years)
    
    diff = current_year - latest
    
    # recent → high score
    if diff <= 1:
        return 1.0
    elif diff <= 2:
        return 0.8
    elif diff <= 4:
        return 0.5
    else:
        return 0.2

In [ ]:
def contextual_adjustment(base_score, jd_struct, res_struct, resume_text):
    
    # 1️⃣ Domain boost
    dom = domain_boost(jd_struct["domain"], res_struct["domain"])
    
    # 2️⃣ Recency boost
    rec = recency_boost(resume_text)
    
    # 🔥 Combine boosts
    boost = 0.1 * dom + 0.1 * rec   # max +0.2
    
    adjusted = base_score + boost
    
    return round(min(adjusted, 1.0), 3)

In [ ]:
results = []

for i, (resume, struct, vec) in enumerate(zip(resume_text_list, Resume_struct, resume_vectors)):
    
    name = extract_name(resume)
    
    skill = skill_score(JD_struct["skills"], struct["skills"], model)
    exp = experience_score(JD_struct["experience"], struct["experience"])
    project = project_score(jd_text, resume, model)
    edu = education_score(jd_text, JD_struct, resume, struct)
    semantic = semantic_similarity(jd_vector, vec)
    
    base = final_score(skill, exp, project, edu, semantic)
    
    # 🔥 Apply contextual boosting
    adjusted = contextual_adjustment(base, JD_struct, struct, resume)
    
    results.append({
        "name": name,
        "base_score": base,
        "adjusted_score": adjusted
    })

In [ ]:
results = sorted(results, key=lambda x: x["adjusted_score"], reverse=True)

for r in results:
    print(f"\n👤 {r['name']}")
    print("-" * 40)
    print(f"Base Score     : {r['base_score']}")
    print(f"Adjusted Score : {r['adjusted_score']}")

# Ranking Engine

In [ ]:
ranked_results = sorted(
    results,
    key=lambda x: x["adjusted_score"],
    reverse=True
)

In [ ]:
for i, r in enumerate(ranked_results):
    r["rank"] = i + 1

In [ ]:
print("\n🏆 FINAL RANKED CANDIDATES\n")

for r in ranked_results:
    print(f"\nRank #{r['rank']} 👤 {r['name']}")

# Gap Analysis

In [ ]:
def gap_analysis(jd_skills, resume_skills):
    jd_set = set(jd_skills)
    res_set = set(resume_skills)
    
    matched = list(jd_set.intersection(res_set))
    missing = list(jd_set - res_set)
    
    return matched, missing

In [ ]:
def semantic_gap_analysis(jd_skills, resume_skills, model, threshold=0.6):
    matched = []
    missing = []
    
    if not jd_skills or not resume_skills:
        return [], jd_skills

    jd_embeddings = model.encode(jd_skills)
    resume_embeddings = model.encode(resume_skills)

    for i, jd_skill in enumerate(jd_skills):
        similarities = [
            util.cos_sim(jd_embeddings[i], res_emb).item()
            for res_emb in resume_embeddings
        ]
        
        if max(similarities) >= threshold:
            matched.append(jd_skill)
        else:
            missing.append(jd_skill)

    return matched, missing

In [ ]:
results = []

for i, (resume, struct, vec) in enumerate(zip(resume_text_list, Resume_struct, resume_vectors)):
    
    name = extract_name(resume)
    
    # Scores
    skill = skill_score(JD_struct["skills"], struct["skills"], model)
    exp = experience_score(JD_struct["experience"], struct["experience"])
    project = project_score(jd_text, resume, model)
    edu = education_score(jd_text, JD_struct, resume, struct)
    semantic = semantic_similarity(jd_vector, vec)
    
    base = final_score(skill, exp, project, edu, semantic)
    adjusted = contextual_adjustment(base, JD_struct, struct, resume)
    
    # 🔥 GAP ANALYSIS (semantic)
    matched, missing = semantic_gap_analysis(
        JD_struct["skills"],
        struct["skills"],
        model
    )
    
    results.append({
        "name": name,
        "adjusted_score": adjusted,
        "matched_skills": matched,
        "missing_skills": missing
    })

In [ ]:
ranked_results = sorted(results, key=lambda x: x["adjusted_score"], reverse=True)

for i, r in enumerate(ranked_results):
    r["rank"] = i + 1

In [ ]:
for r in ranked_results:
    print(f"\n🏅 #{r['rank']} | {r['name']} | Score: {r['adjusted_score']}")
    print(f"✔ Matched: {', '.join(r['matched_skills'])}")
    print(f"✘ Missing: {', '.join(r['missing_skills'][:5])}")

# Reason Generation

In [ ]:
def generate_explanation(result):
    
    name = result["name"]
    score = result["adjusted_score"]
    matched = result.get("matched_skills", [])
    missing = result.get("missing_skills", [])
    
    # pick top items
    top_matched = ", ".join(matched[:3]) if matched else "relevant skills"
    top_missing = ", ".join(missing[:2]) if missing else ""
    
    # 🔥 Build explanation
    explanation = f"{name} is a strong candidate with a score of {score}. "
    
    explanation += f"They demonstrate expertise in {top_matched}, which aligns well with the job requirements. "
    
    if top_missing:
        explanation += f"However, they lack experience in {top_missing}, which could be improved."
    else:
        explanation += "They closely match all key required skills."
    
    return explanation

In [ ]:
for r in ranked_results:
    r["reason"] = generate_explanation(r)

In [ ]:
print("\n🏆 FINAL RESULTS WITH EXPLANATION\n")

for r in ranked_results:
    print(f"\n🏅 Rank #{r['rank']} 👤 {r['name']}")
    print("=" * 60)
    
    print(f"Score: {r['adjusted_score']}")
    
    print("\n🧠 Explanation:")
    print(r["reason"])
    
    print("\n✔ Matched:", ", ".join(r["matched_skills"][:5]))
    print("✘ Missing:", ", ".join(r["missing_skills"][:5]))
    
    print("-" * 60)